In [1]:
# Environment Setup
import os 
import sys

REPO_NAME = "A_Blockchain_Enabled_Framework_for_Misinformation_Monitoring"
REPO_URL = "https://github.com/SnowWoofer/A_Blockchain_Enabled_Framework_for_Misinformation_Monitoring.git" # could make .env or dynamic

if os.path.exists("/kaggle"):
    BASE_DIR = f"/kaggle/working/{REPO_NAME}"
    os.environ["DATA_DIR"] = f"/kaggle/working/{REPO_NAME}"
elif os.path.exists("/content"):    
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = "/content/drive/MyDrive/blockchain_misinformation_data"
    os.makedirs(os.environ["DATA_DIR"], exist_ok=True)
    BASE_DIR = f"/content/{REPO_NAME}"
else:
    BASE_DIR = "."

if not os.path.exists(BASE_DIR):
    !git clone {REPO_URL} {REPO_NAME}

os.chdir(BASE_DIR)
sys.path.append(os.path.join(BASE_DIR, "src"))

Cloning into 'A_Blockchain_Enabled_Framework_for_Misinformation_Monitoring'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 119 (delta 65), reused 82 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 44.40 KiB | 7.40 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [2]:
import importlib
for pkg in ["torch", "transformers", "datasets", "evaluate", "sklearn"]:
    try:
        importlib.import_module(pkg)
        print(f"{pkg}: OK")
    except ImportError:
        print(f"{pkg}: MISSING")
        
!pip install evaluate

torch: OK
transformers: OK
datasets: OK
evaluate: MISSING
sklearn: OK
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00


In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.__version__)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Check Settings > Accelerator > GPU in the Kaggle sidebar.")

CUDA available: True
2.10.0+cu128
True
GPU: Tesla T4


In [4]:
import pandas as pd
import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

seed = 100
np.random.seed(seed)
torch.manual_seed(seed)

In [5]:
df = pd.read_csv("/kaggle/input/datasets/joshcloete/twitter-data-enganon/twitter_data_eng_raw_anon.csv", encoding="utf-8")
df = df.dropna(subset=["text", "label"]).reset_index(drop=True)

train_df = df[df["set"] == "train"][["text", "label"]].reset_index(drop=True)
test_df = df[df["set"] == "test"][["text", "label"]].reset_index(drop=True)

print(f"train: {len(train_df)} rows, test: {len(test_df)} rows")
print(train_df["label"].value_counts())

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train: 91738 rows, test: 10204 rows
label
0    60308
1    31430
Name: count, dtype: int64


In [6]:
model_path = "Davlan/afro-xlmr-large"

id2label = {0: "not misinformation", 1: "misinformation"}
label2id = {"not misinformation": 0, "misinformation": 1}

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

config.json:   0%|          | 0.00/714 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# Freeze all base model params except the pooling layer (near the classification head)
for name, param in model.base_model.named_parameters():
    param.requires_grad = "pooler" in name

In [8]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/91738 [00:00<?, ? examples/s]

Map:   0%|          | 0/10204 [00:00<?, ? examples/s]

In [9]:
accuracy_metric = evaluate.load("accuracy")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    recall = recall_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {
        "accuracy": acc["accuracy"],
        "recall":recall["recall"],
            "f1": f1["f1"], 
    }

In [10]:
sanity_train = tokenized_train.select(range(min(100, len(tokenized_train))))
sanity_test = tokenized_test.select(range(min(50, len(tokenized_test))))

sanity_args = TrainingArguments(
    output_dir="/kaggle/working/sanity_check",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=5,
    report_to="none",
)

sanity_trainer = Trainer(
    model=model,
    args=sanity_args,
    train_dataset=sanity_train,
    eval_dataset=sanity_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

sanity_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Recall,F1
1,1.531250,1.247070,0.780000,0.692308,0.786731


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=7, training_loss=2.156529017857143, metrics={'train_runtime': 7.9623, 'train_samples_per_second': 12.559, 'train_steps_per_second': 0.879, 'total_flos': 93193136332800.0, 'train_loss': 2.156529017857143, 'epoch': 1.0})

In [11]:
import os
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="/kaggle/working/afroxlmr_eng_baseline",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Check for a previous checkpoint to resume from
output_dir = training_args.output_dir
resume_checkpoint = None
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[1]))
        resume_checkpoint = os.path.join(output_dir, checkpoints[-1])
        print(f"Resuming from: {resume_checkpoint}")
    else:
        print("No checkpoint found, starting fresh")

trainer.train(resume_from_checkpoint=resume_checkpoint)

No checkpoint found, starting fresh


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Recall,F1
1,0.892266,0.653320,0.881321,0.751967,0.878824
2,0.679399,0.517090,0.899941,0.775867,0.897704
3,0.646919,0.466553,0.903469,0.796852,0.901825
4,0.592754,0.436523,0.913171,0.821043,0.911963
5,0.596211,0.427246,0.914151,0.816672,0.912798
6,0.576692,0.398193,0.920129,0.847275,0.919401
7,0.568394,0.393066,0.922481,0.851064,0.921781
8,0.542080,0.375732,0.924049,0.856310,0.923427
9,0.547852,0.362061,0.928263,0.857476,0.927570
10,0.445442,0.364014,0.927969,0.846400,0.927042


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=43005, training_loss=0.5661809081463547, metrics={'train_runtime': 30189.022, 'train_samples_per_second': 45.582, 'train_steps_per_second': 1.425, 'total_flos': 1.282362485103297e+18, 'train_loss': 0.5661809081463547, 'epoch': 15.0})

In [12]:
results = trainer.evaluate()
print(results)

trainer.save_model("/kaggle/working/afroxlmr_eng_baseline_final")
tokenizer.save_pretrained("/kaggle/working/afroxlmr_eng_baseline_final")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.332275390625, 'eval_accuracy': 0.9334574676597412, 'eval_recall': 0.8770037889828038, 'eval_f1': 0.9330381244600314, 'eval_runtime': 143.2274, 'eval_samples_per_second': 71.243, 'eval_steps_per_second': 2.227, 'epoch': 15.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/afroxlmr_eng_baseline_final/tokenizer_config.json',
 '/kaggle/working/afroxlmr_eng_baseline_final/tokenizer.json')